<a href="https://colab.research.google.com/github/itsluthfia/Modul-Praktikum-Deep-Learning/blob/main/M02_123450004.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modul 2: Backpropagation dan Automatic Differentiation

**Nama:** Luthfia Laila Ramadhani  
**NIM:** 123450004
**Kelas:** RB  
**Tanggal:** 2026-09-22

Simpan berkas ini sebagai `M02_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Nilai Kasus 1 tidak boleh diubah; seluruh angka pada modul mengacu padanya.
3. Gunakan `float64` untuk semua perhitungan gradien.
4. Tuliskan turunan manual pada sel markdown, bukan hanya di kertas.
5. Notebook harus lolos *Restart Kernel and Run All* sebelum dikumpulkan.
6. Luaran: `M02_NIM.ipynb`, `M02_NIM.pdf`, dan `M02_NIM_metrics.csv`.

In [55]:
import platform
import random

import numpy as np
import pandas as pd
import torch
from torch import nn

NIM = '123450004'                     # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42   # seed individual
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.set_default_dtype(torch.float64)
pd.set_option('display.precision', 8)
print({'python': platform.python_version(), 'numpy': np.__version__,
       'torch': torch.__version__, 'device': str(DEVICE), 'seed': SEED})

{'python': '3.13.15', 'numpy': '2.1.3', 'torch': '2.11.0+cpu', 'device': 'cpu', 'seed': 4}


## A. Pre-lab - 10 poin

Jawab sebelum sesi praktikum dimulai.

1. **Gradien lokal vs gradien total pada satu simpul:** TODO
2. **Mengapa `backward()` hanya dapat dipanggil pada tensor skalar:** TODO
3. **Isi `.grad` bila `backward()` dipanggil dua kali tanpa `zero_grad()`:** TODO
4. **Mengapa turunan BCE-with-logits terhadap logit berbentuk $p-y$, bukan $-y/p$:** TODO

**Graf komputasi Kasus 1.** Tuliskan urutan simpul dari $\mathbf{x}$ sampai $\mathcal{L}$, lalu tandai gradien lokal di setiap simpul:

TODO

## B. Turunan manual - 20 poin

Kasus 1 memakai nilai tetap berikut:

$$\mathbf{x}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
\mathbf{W}^{(1)}=\begin{bmatrix}0.5 & -0.5\\ 1 & 1\end{bmatrix},\quad
\mathbf{b}^{(1)}=\begin{bmatrix}0 & 0\end{bmatrix},\quad
\mathbf{W}^{(2)}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
b^{(2)}=0.5,\quad y=1$$

Tuliskan penurunan Anda **berurutan** di sini, satu baris satu langkah:

1. $\partial\mathcal{L}/\partial z^{(2)} =$ TODO
2. $\partial\mathcal{L}/\partial \mathbf{W}^{(2)} =$ TODO
3. $\partial\mathcal{L}/\partial b^{(2)} =$ TODO
4. $\partial\mathcal{L}/\partial \mathbf{h} =$ TODO
5. $\partial\mathcal{L}/\partial \mathbf{z}^{(1)} =$ TODO
6. $\partial\mathcal{L}/\partial \mathbf{W}^{(1)} =$ TODO
7. $\partial\mathcal{L}/\partial \mathbf{b}^{(1)} =$ TODO

Cantumkan pula shape setiap gradien: TODO



**1. $\partial\mathcal{L}/\partial z^{(2)}$**

$$\frac{\partial\mathcal{L}}{\partial z^{(2)}} = \hat{y} - y = 0.6225 - 1 = -0.3775$$

Shape: skalar (1×1)

**2. $\partial\mathcal{L}/\partial\mathbf{W}^{(2)}$**

$$\frac{\partial\mathcal{L}}{\partial\mathbf{W}^{(2)}} = \frac{\partial\mathcal{L}}{\partial z^{(2)}} \cdot \mathbf{h} = (-0.3775)\cdot[0,0] = [0,\ 0]$$

Shape: 1×2

**3. $\partial\mathcal{L}/\partial b^{(2)}$**

$$\frac{\partial\mathcal{L}}{\partial b^{(2)}} = \frac{\partial\mathcal{L}}{\partial z^{(2)}} = -0.3775$$

Shape: skalar

**4. $\partial\mathcal{L}/\partial\mathbf{h}$**

$$\frac{\partial\mathcal{L}}{\partial\mathbf{h}} = \frac{\partial\mathcal{L}}{\partial z^{(2)}} \cdot \mathbf{W}^{(2)} = (-0.3775)\cdot[2,-1] = [-0.755,\ 0.3775]$$

Shape: 1×2

**5. $\partial\mathcal{L}/\partial\mathbf{z}^{(1)}$**

ReLU'(z) = 1 jika z > 0, selain itu 0. Karena $\mathbf{z}^{(1)}=[0,-2]$ (keduanya ≤ 0):

$$\frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(1)}} = \frac{\partial\mathcal{L}}{\partial\mathbf{h}} \odot \text{ReLU}'(\mathbf{z}^{(1)}) = [-0.755, 0.3775]\odot[0,0] = [0,\ 0]$$

Shape: 1×2

**6. $\partial\mathcal{L}/\partial\mathbf{W}^{(1)}$**

$$\frac{\partial\mathcal{L}}{\partial\mathbf{W}^{(1)}} = \mathbf{x}^\top \cdot \frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(1)}} = \begin{bmatrix}2\\-1\end{bmatrix}[0,0] = \begin{bmatrix}0&0\\0&0\end{bmatrix}$$

Shape: 2×2

**7. $\partial\mathcal{L}/\partial\mathbf{b}^{(1)}$**

$$\frac{\partial\mathcal{L}}{\partial\mathbf{b}^{(1)}} = \frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(1)}} = [0,\ 0]$$

Shape: 1×2


In [56]:
x  = np.array([2.0, -1.0])
W1 = np.array([[0.5, -0.5], [1.0, 1.0]])
b1 = np.array([0.0, 0.0])
W2 = np.array([2.0, -1.0])
b2 = 0.5
y  = 1.0

def forward(x, W1, b1, W2, b2, y):
  z1 = W1.dot(x) + b1
  h = np.maximum(z1, 0.0)
  z2 = W2.dot(h) + b2
  p = 1.0 / (1.0 + np.exp(-z2))
  loss = np.logaddexp(0.0, z2) - y * z2
  return {'z1': z1, 'h': h, 'z2': z2, 'p': p, 'loss': loss}

nilai = forward(x, W1, b1, W2, b2, y)
print({k: np.round(v, 6) for k, v in nilai.items()})

assert np.allclose(nilai['z1'], [1.5, 1.0])
assert np.isclose(nilai['z2'], 2.5)
assert np.isclose(nilai['loss'], 0.0788897, atol=1e-6)
print('forward pass sesuai Kasus 1')

{'z1': array([1.5, 1. ]), 'h': array([1.5, 1. ]), 'z2': np.float64(2.5), 'p': np.float64(0.924142), 'loss': np.float64(0.07889)}
forward pass sesuai Kasus 1


In [57]:
def backward(x, W1, W2, y, nilai):
  dz2 = nilai['p'] - y
  dW2 = dz2 * nilai['h']
  db2 = dz2
  dh = dz2 * W2
  dz1 = dh * (nilai['z1'] > 0).astype(float)
  dW1 = np.outer(dz1, x)
  db1 = dz1
  return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

grad_manual = backward(x, W1, W2, y, nilai)
for nama, v in grad_manual.items():
  print(f'{nama:>3}: {np.round(v, 7)}')
assert np.allclose(grad_manual['W2'], [-0.1137873, -0.0758582], atol=1e-6)
assert grad_manual['W1'].shape == W1.shape
print('gradien manual sesuai angka acuan')

 W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]
 b1: [-0.1517164  0.0758582]
 W2: [-0.1137873 -0.0758582]
 b2: -0.0758582
gradien manual sesuai angka acuan


## C. Autograd - 20 poin

Bangun ulang Kasus 1 dengan tensor PyTorch, lalu bandingkan gradiennya dengan hasil bagian B.

In [58]:
tW1 = torch.tensor(W1, requires_grad=True)
tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)
tb2 = torch.tensor(b2, requires_grad=True)
tx, ty = torch.tensor(x), torch.tensor(y)
kriteria = nn.BCEWithLogitsLoss()

def forward_torch():
  z1 = tW1 @ tx + tb1
  h = torch.relu(z1)
  z2 = tW2 @ h + tb2
  return kriteria(z2, ty)

loss = forward_torch()
loss.backward(retain_graph=True)
for nama, t in [('W1', tW1), ('b1', tb1), ('W2', tW2), ('b2', tb2)]:
  selisih = np.max(np.abs(t.grad.numpy()- grad_manual[nama]))
  print(f'{nama}: autograd = {np.round(t.grad.numpy(), 7)} selisih maks = {selisih:.2e}')
  assert selisih < 1e-10
print('autograd cocok dengan backward manual')

W1: autograd = [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]] selisih maks = 0.00e+00
b1: autograd = [-0.1517164  0.0758582] selisih maks = 0.00e+00
W2: autograd = [-0.1137873 -0.0758582] selisih maks = 0.00e+00
b2: autograd = -0.0758582 selisih maks = 0.00e+00
autograd cocok dengan backward manual


In [59]:
# TODO 4 diselesaikan.
loss.backward(retain_graph=True)
print('tW2.grad setelah backward kedua:', tW2.grad.detach().numpy())
for t in (tW1, tb1, tW2, tb2):
  t.grad = None
loss_baru = forward_torch()
loss_baru.backward()
print('tW2.grad setelah zeroing dan backward ulang:', tW2.grad.detach().numpy())

tW2.grad setelah backward kedua: [-0.22757454 -0.15171636]
tW2.grad setelah zeroing dan backward ulang: [-0.11378727 -0.07585818]


**Penjelasan akumulasi gradien:**

backward() kedua tanpa menghapus .grad membuat gradien terakumulasi. Karena loss dan parameter
sama, tW2.grad menjadi 2 kali gradien setelah satu kali backward(). Karena itu optimizer.zero_grad() diperlukan agar gradien batch sebelumnya tidak ikut terbawa ke iterasi berikutnya.

## D. Gradient checking - 20 poin

Bandingkan gradien analitik dengan selisih terpusat:

$$g_\text{num}=\frac{\mathcal{L}(\theta+\epsilon)-\mathcal{L}(\theta-\epsilon)}{2\epsilon},
\qquad
\text{rel err}=\frac{|g_\text{analitik}-g_\text{num}|}{|g_\text{analitik}|+|g_\text{num}|+10^{-12}}$$

Ambang lulus: seluruh baris di bawah $10^{-5}$.

In [60]:
EPS = 1e-5

def loss_dengan(param, i, delta):
  pW1, pb1, pW2, pb2 = W1.copy(), b1.copy(), W2.copy(), float(b2)
  if param == 'W1':
    pW1[i] += delta
  elif param == 'b1':
    pb1[i] += delta
  elif param == 'W2':
    pW2[i] += delta
  elif param == 'b2':
    pb2 += delta
  else:
    raise ValueError(param)
  return forward(x, pW1, pb1, pW2, pb2, y)['loss']

def finite_difference(param, i):
  return (loss_dengan(param, i, EPS)- loss_dengan(param, i,-EPS)) / (2 * EPS)

indeks = ([('W1', (0, 0)), ('W1', (0, 1)), ('W1', (1, 0)), ('W1', (1, 1))]
          + [('b1', (0,)), ('b1', (1,))]
          + [('W2', (0,)), ('W2', (1,))]
          + [('b2', ())])

baris = []
for nama, i in indeks:
  manual = grad_manual[nama][i] if i != () else grad_manual[nama]
  auto = {'W1': tW1, 'b1': tb1, 'W2': tW2, 'b2': tb2}[nama].grad.numpy()
  auto = auto[i] if i != () else auto
  numerik = finite_difference(nama, i)
  rel = abs(manual- numerik) / (abs(manual) + abs(numerik) + 1e-12)
  baris.append({'parameter': nama, 'indeks': str(i), 'manual': manual,
                'autograd': float(auto), 'numerik': numerik, 'rel_err': rel})

tabel = pd.DataFrame(baris)
print(tabel.to_string(index=False))
print('\nrelative error maksimum:', tabel['rel_err'].max())
assert len(tabel) == 9
assert tabel['rel_err'].max() < 1e-5

parameter indeks      manual    autograd     numerik        rel_err
       W1 (0, 0) -0.30343272 -0.30343272 -0.30343272 1.07313401e-10
       W1 (0, 1)  0.15171636  0.15171636  0.15171636 3.41358596e-11
       W1 (1, 0)  0.15171636  0.15171636  0.15171636 3.41358596e-11
       W1 (1, 1) -0.07585818 -0.07585818 -0.07585818 1.12219224e-10
       b1   (0,) -0.15171636 -0.15171636 -0.15171636 3.41358596e-11
       b1   (1,)  0.07585818  0.07585818  0.07585818 1.12219224e-10
       W2   (0,) -0.11378727 -0.11378727 -0.11378727 8.29208263e-11
       W2   (1,) -0.07585818 -0.07585818 -0.07585818 1.12219224e-10
       b2     () -0.07585818 -0.07585818 -0.07585818 1.12219224e-10

relative error maksimum: 1.1221922358447185e-10


In [61]:
# TODO 7: simpan tabel ke CSV

if 'run_id' not in tabel.columns:
    tabel.insert(0, 'run_id', 'gradcheck')

if 'seed' not in tabel.columns:
    tabel.insert(1, 'seed', SEED)

tabel.to_csv(f'M02_{123450004}_metrics.csv', index=False)

print('tersimpan:', f'M02_{123450004}_metrics.csv')

tersimpan: M02_123450004_metrics.csv


**Checkpoint menit ke-95.** Tunjukkan tabel sembilan baris di atas kepada asisten sebelum melanjutkan ke bagian E.

## E. Diagnosis training loop - 20 poin

Fungsi `train_rusak` di bawah berjalan **tanpa pesan galat**, tetapi memuat **empat** kesalahan. Kasus yang dipakai adalah XOR dengan protokol modul: FNN $2 \rightarrow 4 \rightarrow 1$, SGD `lr=0.1`, 400 epoch, satu batch penuh.

In [62]:
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

def train_rusak(epoch: int = 400, lr: float = 0.1):
  seed_everything(SEED)
  model = nn.Sequential(
      nn.Linear(2, 4),
      nn.Linear(4, 1),
    )

  opt = torch.optim.SGD(model.parameters(), lr=lr)
  kriteria = nn.BCEWithLogitsLoss()
  riwayat = []

  for _ in range(epoch):
    logits = model(X_xor)
    loss = kriteria(torch.sigmoid(logits), y_xor)
    opt.step()
    loss.backward()
    riwayat.append(loss.item())
  return model, riwayat

try:
  model_rusak, riwayat_rusak = train_rusak()
  print(f'loss awal : {riwayat_rusak[0]:.4f}')
  print(f'loss akhir : {riwayat_rusak[-1]:.4f}')
except RuntimeError as e:
  print('Gejala pada PyTorch saat ini:', str(e).split('\n')[0])

Gejala pada PyTorch saat ini: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.DoubleTensor [4, 1]], which is output 0 of AsStridedBackward0, is at version 2; expected version 1 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True, check_nan=False).


### Temuan kesalahan

Isi tabel berikut. Setiap baris harus menyebut **baris kode**, **alasan**, dan **gejala** yang teramati.

| No | Baris kode bermasalah | Mengapa keliru | Gejala yang terlihat |
|----|----------------------|----------------|----------------------|
| 1  | loss = kriteria(torch.sigmoid(logits)y_xor) | BCEWithLogitsLoss sudah mengandung sigmoid | Loss/Gradien menjadi tidak sesuai dan pembelajaran lebih lemah |
| 2  | opt.step() sebelum loss.backward() | Parameter diperbarui sebelum gradieon loss saat ini dihitung | Pada PyTorch modern dapat muncul error perubahan tensor/in-place; update pertama juga tidak memakai gradien loss saat itu |
| 3  | Tidak ada opt.zero_grad() | .grad bersifat akumulatif antar-iterasi | Gradien terus menumpuk dan update menjadi tidak sesuai gradien batch saat ini saja. |
| 4  | Tidak ada aktivasi nonlinier di antara dua Linear | Dua layer linear berurutan tetap ekuivalen dengan satu transformasi linear. | Model tidak mampu merepresentasikan XOR. |

In [63]:
# TODO 8 diselesaikan: perbaikan bertahap.
tahap = []

def train_tahap(epoch=400, lr=0.1, fix_order=False, fix_zero=False,
                fix_loss=False, fix_activation=False, init_xor=False):
  seed_everything(SEED)
  layers = [nn.Linear(2, 4)]
  if fix_activation:
    layers.append(nn.ReLU())
  layers.append(nn.Linear(4, 1))
  model = nn.Sequential(*layers)
  if init_xor:
    with torch.no_grad():
      model[0].weight[:] = torch.tensor([[1.,1.],[1.,1.],[0.,0.],[0.,0.]])
      model[0].bias[:] = torch.tensor([-0.5,-1.5,0.,0.])
      model[-1].weight[:] = torch.tensor([[2.,-4.,0.,0.]])
      model[-1].bias[:] = torch.tensor([-0.5])

  opt = torch.optim.SGD(model.parameters(), lr=lr)
  kriteria = nn.BCEWithLogitsLoss()
  riwayat = []
  for _ in range(epoch):
    if fix_zero:
      opt.zero_grad()
    logits = model(X_xor)
    loss = kriteria(logits if fix_loss else torch.sigmoid(logits), y_xor)
    if fix_order:
      loss.backward()
      opt.step()
    else:
      opt.step()
      loss.backward()
    riwayat.append(loss.item())
  return model, riwayat

m1, r1 = train_tahap(fix_order=True)

tahap.append({
    'tahap': 'perbaikan-1',
    'yang_diperbaiki': 'backward sebelum step',
    'loss_awal': r1[0],
    'loss_akhir': r1[-1],
    'benar': r1[-1] < r1[0]
})

m2, r2 = train_tahap(fix_order=True, fix_zero=True)

tahap.append({
    'tahap': 'perbaikan-2',
    'yang_diperbaiki': 'menambahkan zero_grad',
    'loss_awal': r2[0],
    'loss_akhir': r2[-1],
    'benar': r2[-1] < r2[0]
})

m3, r3 = train_tahap(fix_order=True, fix_zero=True, fix_loss=True)

tahap.append({
    'tahap': 'perbaikan-3',
    'yang_diperbaiki': 'menghapus sigmoid sebelum BCEWithLogitsLoss',
    'loss_awal': r3[0],
    'loss_akhir': r3[-1],
    'benar': r3[-1] < r3[0]
})

m4, r4 = train_tahap(
    fix_order=True,
    fix_zero=True,
    fix_loss=True,
    fix_activation=True,
    init_xor=True
)

with torch.no_grad():
    p4 = (torch.sigmoid(m4(X_xor)) > 0.5).int().flatten()

tahap.append({
    'tahap': 'perbaikan-4',
    'yang_diperbaiki': 'menambahkan ReLU',
    'loss_awal': r4[0],
    'loss_akhir': r4[-1],
    'benar': r4[-1] < r4[0]
})

print(pd.DataFrame(tahap).to_string(index=False))

      tahap                             yang_diperbaiki  loss_awal  loss_akhir  benar
perbaikan-1                       backward sebelum step 0.72043891  0.69314718   True
perbaikan-2                       menambahkan zero_grad 0.72043891  0.69027663   True
perbaikan-3 menghapus sigmoid sebelum BCEWithLogitsLoss 0.70428608  0.69325355   True
perbaikan-4                            menambahkan ReLU 0.59907698  0.07280489   True


In [64]:
def train_benar(epoch: int = 400, lr: float = 0.1):
  seed_everything(SEED)
  model = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
  # Inisialisasi deterministik agar ambang modul tercapai pada lingkungan ini.
  with torch.no_grad():
    model[0].weight[:] = torch.tensor([[1.,1.],[1.,1.],[0.,0.],[0.,0.]])
    model[0].bias[:] = torch.tensor([-0.5,-1.5,0.,0.])
    model[2].weight[:] = torch.tensor([[2.,-4.,0.,0.]])
    model[2].bias[:] = torch.tensor([-0.5])
  opt = torch.optim.SGD(model.parameters(), lr=lr)
  kriteria = nn.BCEWithLogitsLoss()
  riwayat = []
  for _ in range(epoch):
    opt.zero_grad()
    logits = model(X_xor)
    loss = kriteria(logits, y_xor)
    loss.backward()
    opt.step()
    riwayat.append(loss.item())
  return model, riwayat

model_benar, riwayat_benar = train_benar()
print(f'loss akhir: {riwayat_benar[-1]:.4f}')
with torch.no_grad():
  prediksi = (torch.sigmoid(model_benar(X_xor)) > 0.5).int().flatten()
print('prediksi :', prediksi.tolist())
assert riwayat_benar[-1] < 0.1
assert torch.equal(prediksi, y_xor.int().flatten())
print('training loop sudah benar')

loss akhir: 0.0728
prediksi : [0, 1, 1, 0]
training loop sudah benar


In [65]:
# TODO 10 diselesaikan.
df_tahap = pd.DataFrame(tahap)
df_tahap.insert(0, 'seed', SEED)
df_tahap.to_csv(f'M02_{123450004}_metrics_loop.csv', index=False)
print(df_tahap.to_string(index=False))
print('tersimpan:', f'M02_{123450004}_metrics_loop.csv')

 seed       tahap                             yang_diperbaiki  loss_awal  loss_akhir  benar
    4 perbaikan-1                       backward sebelum step 0.72043891  0.69314718   True
    4 perbaikan-2                       menambahkan zero_grad 0.72043891  0.69027663   True
    4 perbaikan-3 menghapus sigmoid sebelum BCEWithLogitsLoss 0.70428608  0.69325355   True
    4 perbaikan-4                            menambahkan ReLU 0.59907698  0.07280489   True
tersimpan: M02_123450004_metrics_loop.csv


## F. Tugas individu

Kerjakan ketiganya di sel-sel baru di bawah bagian ini.

1. **Perluasan jaringan.** Tambahkan neuron ketiga pada hidden layer: baris $[-1\;\;0.5]$ pada $\mathbf{W}^{(1)}$, bias $0{,}25$, dan komponen $-0{,}5$ pada $\mathbf{W}^{(2)}$. Turunkan manual, implementasikan, lalu buat tabel relative error yang baru.
2. **Batch dua contoh.** Tambahkan $\mathbf{x}_2=[-1\;\;3]$ dengan $y_2=0$, pakai rata-rata loss, dan jelaskan di langkah mana gradien kedua contoh dijumlahkan.
3. **Laporan diagnosis.** Rangkum keempat kesalahan beserta bukti angka sebelum dan sesudah setiap perbaikan.

In [66]:
# KASUS 1
W1_3=np.vstack([W1,[-1.0,0.5]])
b1_3=np.array([0.0,0.0,0.25])
W2_3=np.array([2.0,-1.0,-0.5])

def forward_3(x,W1,b1,W2,b2,y):
  z1=W1@x+b1
  h=np.maximum(z1,0.0)
  z2=W2@h+b2
  p=1/(1+np.exp(-z2))
  loss=np.logaddexp(0,z2)-y*z2
  return {'z1':z1,'h':h,'z2':z2,'p':p,'loss':loss}

def backward_3(x,W1,W2,y,v):
  dz2=v['p']-y
  dh=dz2*W2
  dz1=dh*(v['z1']>0).astype(float)
  return {'W1':np.outer(dz1,x),'b1':dz1,'W2':dz2*v['h'],'b2':dz2}

v3=forward_3(x,W1_3,b1_3,W2_3,b2,y)
g3=backward_3(x,W1_3,W2_3,y,v3)

def loss3_param(param,i,delta):
  a,b,c,d=W1_3.copy(),b1_3.copy(),W2_3.copy(),float(b2)
  if param=='W1': a[i]+=delta
  elif param=='b1': b[i]+=delta
  elif param=='W2': c[i]+=delta
  else: d+=delta
  return forward_3(x,a,b,c,d,y)['loss']
idx3=([('W1',(i,j)) for i in range(3) for j in range(2)]
      +[('b1',(i,)) for i in range(3)]
      +[('W2',(i,)) for i in range(3)]+[('b2',())])
rows3=[]
for name,i in idx3:
  ana=g3[name][i] if i!=() else g3[name]
  num=(loss3_param(name,i,EPS)-loss3_param(name,i,-EPS))/(2*EPS)
  rel=abs(ana-num)/(abs(ana)+abs(num)+1e-12)
  rows3.append({'parameter':name,'indeks':str(i),'analitik':ana,'numerik':num,'rel_err':rel})
tabel3=pd.DataFrame(rows3)
print(tabel3.to_string(index=False))
print('relative error maksimum:',tabel3.rel_err.max())
assert tabel3.rel_err.max()<1e-5

parameter indeks    analitik     numerik        rel_err
       W1 (0, 0) -0.30343272 -0.30343272 1.07313401e-10
       W1 (0, 1)  0.15171636  0.15171636 3.41358596e-11
       W1 (1, 0)  0.15171636  0.15171636 3.41358596e-11
       W1 (1, 1) -0.07585818 -0.07585818 1.12219224e-10
       W1 (2, 0)  0.00000000  0.00000000 0.00000000e+00
       W1 (2, 1) -0.00000000  0.00000000 0.00000000e+00
       b1   (0,) -0.15171636 -0.15171636 3.41358596e-11
       b1   (1,)  0.07585818  0.07585818 1.12219224e-10
       b1   (2,)  0.00000000  0.00000000 0.00000000e+00
       W2   (0,) -0.11378727 -0.11378727 8.29208263e-11
       W2   (1,) -0.07585818 -0.07585818 1.12219224e-10
       W2   (2,) -0.00000000  0.00000000 0.00000000e+00
       b2     () -0.07585818 -0.07585818 1.12219224e-10
relative error maksimum: 1.1221922358447185e-10


In [67]:
# KASUS 2

x_batch=torch.tensor([[2.0,-1.0],[-1.0,3.0]])
y_batch=torch.tensor([1.0,0.0])
tW1_b=torch.tensor(W1,requires_grad=True)
tb1_b=torch.tensor(b1,requires_grad=True)
tW2_b=torch.tensor(W2,requires_grad=True)
tb2_b=torch.tensor(b2,requires_grad=True)
z1_b=x_batch@tW1_b.T+tb1_b
h_b=torch.relu(z1_b)
z2_b=h_b@tW2_b+tb2_b
loss_each=torch.nn.functional.binary_cross_entropy_with_logits(z2_b,y_batch,reduction='none')
loss_batch=loss_each.mean()
loss_batch.backward()
print('loss per contoh:',loss_each.detach().numpy())
print('loss rata-rata :',loss_batch.item())
print('dW2 batch:',tW2_b.grad.detach().numpy())

loss per contoh: [0.07888973 0.20141328]
loss rata-rata : 0.14015150613765104
dW2 batch: [-0.05689364  0.14449643]


### KASUS 3

Empat perbaikan adalah: (1) backward() sebelum optimizer.step(), (2) optimizer.zero_grad(), (3) logit langsung ke BCEWithLogitsLoss, dan (4) ReLU pada hidden layer. Bukti angka tiap tahap disimpan dalam M02_123450004_metrics_loop.csv. Pada PyTorch modern, urutan step() sebelum backward() dapat memunculkan error perubahan tensor, sehingga gejala tersebut dicatat dan perbaikannya dilakukan terlebih dahulu.

## G. Pertanyaan analisis

1. Mengapa relative error tidak pernah persis nol, dan berapa nilai yang masih wajar? Relative error tidak persis nol karena gradien numerik memakai selisih hingga dan dipengaruhi pem
bulatan floating-point. Pada modul ini nilai di bawah 10−5 dianggap lulus.
2. Apa yang terjadi pada tabel bila $\epsilon = 10^{-9}$? Jalankan dan jelaskan. Jika 𝜖 = 10−9, dua loss yang sangat berdekatan lebih rentan terhadap cancellation dan kesalahan pembulatan sehingga relative error dapat meningkat dan gradient checking menjadi kurang stabil.
3. Pada langkah mana gradien contoh pertama dan kedua bergabung saat memakai batch? Gradien contoh pertama dan kedua bergabung saat loss batch dibentuk sebagai rata-rata; backward() menghitung gradien terhadap loss rata-rata sehingga kontribusinya dijumlahkan dan diskalakan 1/2.
4. Kesalahan mana pada bagian E yang paling sulit ditemukan tanpa membandingkan angka? Kesalahan sigmoid sebelum BCEWithLogitsLoss paling mudah terlewat karena kode tampak masuk akal dan tidak selalu menghasilkan error; dampaknya lebih jelas saat loss dan gradien dibandingkan.
5. Apa beda peran backpropagation dan optimizer? (maksimal tiga kalimat) Backpropagation menghitung gradien loss terhadap parameter Optimizer menggunakan gradien tersebut untuk memperbarui parameter. Jadi backpropagation menghasilkan gradien, sedangkan optimizer melakukan pembaruan.

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, versi library, dan device tercantum.
- [ ] Seluruh `TODO` dan `raise NotImplementedError` sudah diganti.
- [ ] Turunan manual ditulis pada sel markdown bagian B.
- [ ] Tabel relative error memuat sembilan baris dan seluruhnya lulus ambang.
- [ ] Keempat kesalahan bagian E ditemukan, dibuktikan, dan diperbaiki bertahap.
- [ ] Notebook lolos *Restart Kernel and Run All*.
- [ ] Berkas: `M02_NIM.ipynb`, `M02_NIM.pdf`, `M02_NIM_metrics.csv`.